# `playground_resnet` — Exact Layer×Head Decomposition of CLIP-ResNet (RN50 / RN101)

Companion notebook to the note *"Exact Per-Input Decomposition of CLIP-ResNet into
Layer×Head Components, with SVD-Based Text Interpretation"*.  It reproduces, for a
**CLIP ResNet** image encoder, the same PCLens pipeline the repo already runs for ViTs —
by writing the ResNet embedding as an **exact per-image sum of components**

$$M_\text{image}(I)=\sum_{l=0}^{L}\sum_{h=1}^{H} c_{l,h}(I)\;+\;\sum_{h=1}^{H} c_{P,h}(I)$$

where $l=0$ is the stem write, $l=1..L$ are the Bottleneck blocks, and $h$ ranges over the
attention-pooling heads.  Post-residual ReLUs are frozen as $0/1$ gates $D_l(I)$ (making the
residual stream linear on this input) and the pooling softmax weights $a^h(I)$ are frozen
(making attention pooling linear in the tokens); both are self-checking numerically.

**Design for reuse.**  Components are stored in the *same on-disk layout* the ViT pipeline
uses — `attn` `[N, L+1, H, d]` = $c_{l,h}$ and `mlp` `[N, 1, d]` = $\sum_h c_{P,h}$ — so that
`embed = attn.sum((1,2)) + mlp.sum(1)` and every existing analysis function
(`get_data`, `svd_data_approx`, `visualize_principal_component`, …) works with `model="RN50"`.

**Sections.**  §0 setup · §A exact decomposition (sum-recovery ✓) · §B dataset run + zero-shot
accuracy · §C ReLU-dead / zero-component fraction · §D mean-ablation orderings · §E PC
characterization (PCLens SVD) · §F PCLens visualization (images/text, both poles) · §G does the
text encoder share heads? cross-encoder head pairs.

## §0 — Setup

Flexible over `RN50` / `RN101` (identical architecture, different depth: `[3,4,6,3]` vs
`[3,4,23,3]`).  TF32 is disabled so the linear contribution split is exact (conv/matmul in
true fp32; on Ampere/Hopper TF32's 10-bit mantissa breaks `conv(a)+conv(b)=conv(a+b)`).

In [ ]:
import os, json, glob, numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image

torch.backends.cudnn.allow_tf32 = False          # exact fp32 for the decomposition
torch.backends.cuda.matmul.allow_tf32 = False

from utils.models.factory import create_model_and_transforms, get_tokenizer
from utils.models.resnet_prs import decompose_resnet_image, verify_decomposition
from utils.scripts.algorithms_text_explanations_funcs import (
    get_data, print_data, visualize_principal_component)

# ---- config -----------------------------------------------------------------
device            = "cuda:0"
model_name        = "RN50"            # "RN50" | "RN101"
pretrained        = "openai"
seed              = 0
cache_dir         = "../cache"
path              = "./datasets/"
O                 = "output_dir"

dataset_image_name = "imagenet"
dataset_text_name  = "top_1500_nouns_5_sentences_imagenet_clean"   # PC-labelling probe set
samples_per_class      = 5            # pilot: ~5000 images
tot_samples_per_class  = 50
num_last_blocks        = 6            # PC-decompose the last N block-components (layer3+layer4)
algorithm              = "svd_data_approx"

In [ ]:
## Load the CLIP ResNet (both towers) + tokenizer.
model, _, preprocess = create_model_and_transforms(
    model_name, pretrained=pretrained, precision="fp32", cache_dir=cache_dir)
model.to(device).eval()
tokenizer = get_tokenizer(model_name)
visual = model.visual

n_blocks = 1 + sum(len(getattr(visual, f"layer{i}")) for i in range(1, 5))   # L+1 (incl. stem)
H = visual.attnpool.num_heads
d = visual.attnpool.c_proj.out_features
K = visual.attnpool.positional_embedding.shape[0] - 1                        # spatial tokens
print(f"{model_name}: L+1 = {n_blocks} block-components (l=0 stem .. {n_blocks-1}), "
      f"H = {H} pooling heads, d = {d}, K = {K} spatial tokens")
print(f"TEXT tower: {len(model.transformer.resblocks)} layers, "
      f"{model.transformer.resblocks[0].attn.num_heads} heads")

## §A — The exact decomposition (foundation)

`decompose_resnet_image(visual, image)` returns `c_lh` `[B, L+1, H, d]` and the content-free
`c_pos` `[B, d]`, with the **exact identity** `c_lh.sum((1,2)) + c_pos == visual(image)`.
The `check=True` flag asserts exactness of *both* stages (feature-map split and attn-pool split).

In [ ]:
files = sorted(glob.glob(f"{path}imagenet/val/*/*.JPEG"))[:8]
imgs  = torch.stack([preprocess(Image.open(f).convert("RGB")) for f in files]).to(device)

out = decompose_resnet_image(visual, imgs, check=True)     # asserts exactness at each stage
c_lh, c_pos = out["c_lh"], out["c_pos"]
print("components c_lh:", tuple(c_lh.shape), " | positional c_pos:", tuple(c_pos.shape),
      " | pooling weights a^h:", tuple(out["attn"].shape))

# ✅ Verify — sum of components recovers the raw image embedding.
with torch.no_grad():
    real = visual(imgs)
recon = c_lh.sum(dim=(1, 2)) + c_pos
err = (recon - real).abs().max().item()
print(f"\n✅ image(I) = Σ_l Σ_h c_(l,h)(I) + Σ_h c_(P,h)(I)   max|recon-real| = {err:.2e} "
      f"(rel {err/real.abs().max().item():.2e})")
assert err < 1e-3 * real.abs().max().item()

## §B — Dataset run + zero-shot accuracy

Precompute the components over ImageNet-val (pilot ≈ 5 img/class) and the zero-shot classifier,
saved in the ViT-compatible layout.  Run once from a shell (fast on GPU):

```bash
# components  ->  {imagenet}_{attn,mlp,labels,embeddings,poolattn}_RN50_seed_0.npy
python -m utils.scripts.compute_activation_values_resnet \
    --model RN50 --pretrained openai --dataset imagenet \
    --samples_per_class 5 --tot_samples_per_class 50 \
    --batch_size 64 --num_workers 0 --device cuda:0 --cache_dir ../cache \
    --output_dir output_dir --seed 0 --max_nr_samples_before_writing 640

# zero-shot classifier  ->  imagenet_classifier_RN50.npy
python -m utils.scripts.compute_classes_embeddings \
    --model RN50 --pretrained openai --dataset imagenet \
    --device cuda:0 --cache_dir ../cache --output_dir output_dir
```
`num_workers 0` avoids fork limits in restricted environments.

In [ ]:
def Ld(p):  # load .npy (mmap) -> float32 tensor on `device`
    return torch.tensor(np.load(p, mmap_mode="r")).to(device, torch.float32)

pref = f"{O}/{dataset_image_name}"
attns      = Ld(f"{pref}_attn_{model_name}_seed_{seed}.npy")        # [N, L+1, H, d] = c_{l,h}
mlps       = Ld(f"{pref}_mlp_{model_name}_seed_{seed}.npy")         # [N, 1, d]      = Σ_h c_{P,h}
labels     = torch.tensor(np.load(f"{pref}_labels_{model_name}_seed_{seed}.npy")).long().to(device)
embeddings = Ld(f"{pref}_embeddings_{model_name}_seed_{seed}.npy")  # [N, d]  (= visual(I))
poolattn   = Ld(f"{pref}_poolattn_{model_name}_seed_{seed}.npy")    # [N, H, K+1]  a^h(I)
classifier = Ld(f"{pref}_classifier_{model_name}.npy")             # [d, C]
N, Lp1, Hh, dd = attns.shape
print(f"N={N} images | attns {tuple(attns.shape)} | mlps {tuple(mlps.shape)} | classes {classifier.shape[1]}")

In [ ]:
@torch.no_grad()
def embed(a, m):                     # reuse the ViT identity: Σ over blocks & heads + positional
    return a.sum(dim=(1, 2)) + m.sum(dim=1)

@torch.no_grad()
def zeroshot(z):
    zc = z / z.norm(dim=-1, keepdim=True)
    return (zc @ classifier).argmax(1).eq(labels).float().mean().item() * 100

# ✅ Verify — on-disk identity + zero-shot accuracy of the reassembled embedding.
print("identity  max|Σcomponents - embeddings| =", (embed(attns, mlps) - embeddings).abs().max().item())
base_acc = zeroshot(embed(attns, mlps))
print(f"{model_name} zero-shot ImageNet top-1 (this subset): {base_acc:.2f}%   (RN50 full-val ≈ 59.6%)")

## §C — ReLU-dead / zero-component fraction

The paper predicts that early/stem writes are heavily attenuated by the long product of
downstream ReLU gates $M_{l\to L}=\prod_k D_k S_k$, so their components carry small norm (and
occasionally vanish exactly).  We quantify, per component $(l,h)$:

* **alive fraction** $\pi_{l,h}=\tfrac1N\#\{n: \lVert c_{l,h}(I_n)\rVert>\varepsilon\}$,
* **exact-zero fraction** (writes annihilated by the gates),
* **mean relative norm** $\lVert c_{l,h}\rVert / \lVert M_\text{image}\rVert$.

In [ ]:
tiny = 1e-6
comp_norm = attns.norm(dim=-1)                                   # [N, L+1, H]
emb_norm  = embeddings.norm(dim=-1, keepdim=True)               # [N, 1]
mean_relnorm = (comp_norm / emb_norm.unsqueeze(-1)).mean(0)     # [L+1, H]
alive_frac   = (comp_norm > tiny).float().mean(0)              # π_{l,h}
exact_zero   = (comp_norm == 0).float().mean(0)               # exactly killed
block_relnorm = (attns.sum(2).norm(dim=-1) / emb_norm).mean(0)  # Σ_h c_{l,h} rel-norm per block

print("exact-zero components (any l,h):", (exact_zero > 0).sum().item(), "of", exact_zero.numel())
print("stem (l=0) mean alive fraction:", alive_frac[0].mean().item(),
      " | stem mean rel-norm:", mean_relnorm[0].mean().item())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
im0 = ax[0].imshow(mean_relnorm.cpu(), aspect="auto", cmap="viridis")
ax[0].set(title="mean ‖c_(l,h)‖ / ‖M_image‖", xlabel="head h", ylabel="block l (0=stem)")
plt.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(alive_frac.cpu(), aspect="auto", cmap="magma", vmin=0, vmax=1)
ax[1].set(title="alive fraction π_(l,h)", xlabel="head h", ylabel="block l")
plt.colorbar(im1, ax=ax[1])
ax[2].plot(block_relnorm.cpu(), marker="o")
ax[2].set(title="per-block relative norm Σ_h c_(l,h)", xlabel="block l (0=stem)",
          ylabel="‖·‖ / ‖M_image‖"); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## §D — Mean-ablation orderings (zero-shot accuracy)

Mean-ablation replaces a component by its dataset mean $c_{l,h}(I)\to\bar c_{l,h}$ (removes the
image-specific signal, keeps the bias).  We ablate **cumulatively** in three orders — most-
important→least, least→most, and random — ranking components by mean norm (a variance proxy),
and track zero-shot accuracy.  This is the ResNet analogue of the ViT completeness/localisation test.

In [ ]:
mu = attns.mean(0, keepdim=True)                                 # [1, L+1, H, d]  per-component mean
comps = [(l, h) for l in range(Lp1) for h in range(H)]
rank = sorted(comps, key=lambda lh: comp_norm[:, lh[0], lh[1]].mean().item(), reverse=True)  # important→

@torch.no_grad()
def cumulative_ablate(order):
    z = embed(attns, mlps).clone()
    accs = [zeroshot(z)]
    for (l, h) in order:
        z = z - (attns[:, l, h] - mu[0, l, h])                  # -> replace by mean
        accs.append(zeroshot(z))
    return accs

acc_top  = cumulative_ablate(rank)                              # top → last
acc_bot  = cumulative_ablate(rank[::-1])                        # last → top
rng = np.random.default_rng(0); ro = rank.copy(); rng.shuffle(ro)
acc_rand = cumulative_ablate(list(ro))                          # random

xs = np.arange(len(comps) + 1)
plt.figure(figsize=(8, 5))
plt.plot(xs, acc_top,  label="most-important first")
plt.plot(xs, acc_bot,  label="least-important first")
plt.plot(xs, acc_rand, label="random", alpha=.7)
plt.axhline(0.1, ls=":", c="grey", label="chance (1/1000)")
plt.xlabel("# components mean-ablated"); plt.ylabel("zero-shot top-1 (%)")
plt.title(f"{model_name}: cumulative mean-ablation by ordering"); plt.legend(); plt.grid(alpha=.3)
plt.show()
print(f"start {acc_top[0]:.1f}%  |  all-ablated {acc_top[-1]:.1f}%")

## §E — PC characterization (PCLens SVD), labelled by text

For each late block-component we mean-center its per-image vectors, run an SVD, keep the top
principal directions, and label them with the probe text set — exactly `svd_data_approx`, reused
verbatim.  Because the saved `attn`/`mlp` arrays are in ViT layout, `compute_text_explanations`
runs unchanged; just set `--num_of_last_layers` to the number of trailing block-components.

```bash
# 1) embed the probe text set  ->  top_1500_nouns_5_sentences_imagenet_clean_RN50.npy
python -m utils.scripts.compute_text_embeddings \
    --model RN50 --pretrained openai \
    --data_path utils/text_descriptions/top_1500_nouns_5_sentences_imagenet_clean.txt \
    --output_dir output_dir --device cuda:0 --cache_dir ../cache

# 2) per-(block,head) SVD + text labels  ->  ..._completeness_..._algo_svd_data_approx_seed_0.jsonl
python -m utils.scripts.compute_text_explanations \
    --model RN50 --dataset imagenet \
    --text_descriptions top_1500_nouns_5_sentences_imagenet_clean \
    --num_of_last_layers 6 --algorithm svd_data_approx \
    --input_dir output_dir --output_dir output_dir --device cuda:0 --seed 0
```

In [ ]:
attention_dataset = (f"{O}/{dataset_image_name}_completeness_{dataset_text_name}_"
                     f"{model_name}_algo_{algorithm}_seed_{seed}.jsonl")
data = get_data(attention_dataset, skip_final=True)
print(f"loaded {len(data)} (block,head,PC) rows from\n  {attention_dataset}\n")
print_data(data, min_princ_comp=3)          # top-3 PCs per late (block,head), with +/- text labels

## §F — PCLens visualization: nearest images **and** texts, both poles

`visualize_principal_component(block, head, pc, ...)` shows the images and probe texts whose CLIP
embeddings align most/least with a chosen principal direction $v_j$ of component $(l,h)$ — the two
poles $\pm v_j$.  Reused unchanged; `block` plays the role of the ViT `layer` index.

In [ ]:
final_embeddings_texts = Ld(f"{O}/{dataset_text_name}_{model_name}.npy")
with open(f"utils/text_descriptions/{dataset_text_name}.txt") as f:
    texts_str = np.array([x.strip() for x in f])

# pick the strongest late component (by variance) to inspect
late = [(l, h) for (l, h) in comps if l >= Lp1 - num_last_blocks]
l_star, h_star = max(late, key=lambda lh: comp_norm[:, lh[0], lh[1]].std().item())
print(f"inspecting block {l_star}, head {h_star}, PC 0")
visualize_principal_component(
    l_star, h_star, 0, 8, 8, 0,
    attention_dataset, embeddings, final_embeddings_texts,
    seed, path, texts_str,
    samples_per_class=samples_per_class, dataset=dataset_image_name,
    tot_samples_per_class=tot_samples_per_class)

## §G — Does the text encoder share heads?  Cross-encoder head pairs

The RN50 **text tower is an ordinary Transformer**, so the existing text PRS decomposes it into
per-(layer,head) components in the same shared space.  We characterise each head by its **leading
mean-centered principal direction** $v_1$ (SVD of its per-input components), then score every
(image pooling-head, text head) pair by $|\cos(v_1^{\text{img}}, v_1^{\text{txt}})|$ and read off
the most aligned pairs — the ResNet analogue of the cross-encoder head-matching experiment (E6).

```bash
# text-tower components over a text set (EOS token), same layout
python -m utils.scripts.compute_activation_values_text \
    --model RN50 --pretrained openai \
    --dataset imagenet_descriptions_personal \
    --device cuda:0 --cache_dir ../cache --output_dir output_dir --seed 0
```

In [ ]:
def leading_dirs(A):
    # A [N, G, d] -> [G, d] unit leading right-singular vector per group (mean-centered)
    G = A.shape[1]; V = torch.zeros(G, A.shape[2], device=A.device)
    Ac = A - A.mean(0, keepdim=True)
    for g in range(G):
        u, s, vh = torch.linalg.svd(Ac[:, g], full_matrices=False)
        V[g] = vh[0]
    return torch.nn.functional.normalize(V, dim=-1)

# image side: flatten the late (block,head) components into groups
img_late = attns[:, Lp1 - num_last_blocks:].reshape(N, -1, d)     # [N, G_img, d]
V_img = leading_dirs(img_late)

try:
    tpref = f"{O}/imagenet_descriptions_personal"
    attns_t = Ld(f"{tpref}_attn_text_{model_name}_seed_{seed}.npy")   # [Nt, l, h, d]
    Nt, lt, ht, _ = attns_t.shape
    txt_late = attns_t[:, lt - num_last_blocks:].reshape(Nt, -1, d)
    V_txt = leading_dirs(txt_late)
    A = (V_img @ V_txt.T).abs()                                      # [G_img, G_txt]
    plt.figure(figsize=(7, 6)); plt.imshow(A.cpu(), aspect="auto", cmap="viridis")
    plt.colorbar(); plt.title("|cos(v1_img, v1_txt)|  head-pair alignment")
    plt.xlabel("text late-head index"); plt.ylabel("image late pool-head index"); plt.show()
    flat = A.flatten(); top = flat.topk(10).indices
    print("Top aligned (image-group, text-group, |cos|):")
    for t in top:
        i, j = t // A.shape[1], t % A.shape[1]
        print(f"  img {i.item():3d}  <->  txt {j.item():3d}   {A[i, j]:.3f}")
except FileNotFoundError:
    print("Run the compute_activation_values_text command above first (text-tower components).")

## Status

* §A–§D are **verified** on the pilot run (exact sum-recovery, zero-shot accuracy, ReLU-dead
  fraction, mean-ablation orderings).
* §E–§G **reuse the existing scripts/functions unchanged** (the ResNet components are stored in
  the ViT `[N, blocks, heads, d]` layout); each has a one-line precompute command shown above.
* Everything is generic over `RN50` / `RN101` — set `model_name` in §0.